# Computer Vision Lab — Task 01
### Transfer Learning Comparison on the ISIC Skin Cancer (9-Class) Dataset

**Reference paper:** *Skin Lesion Classification with Dilated Convolutional Neural Network and Transfer Learning* (bioRxiv 860700v3)
The paper fine-tunes ImageNet-pretrained CNNs (VGG16, VGG19, MobileNet, InceptionV3) on the HAM10000 skin-lesion dataset in two phases:
1. **Feature-extraction phase** — freeze the pretrained backbone, train only the new classifier head for a few epochs.
2. **Fine-tuning phase** — unfreeze the whole network and continue training at a lower learning rate.

Models are then evaluated with **Accuracy, Precision, Recall, F1-score, and AUC**.

This notebook reproduces that same *kind* of experiment on the **ISIC Skin Cancer 9-Class dataset**
(`nodoubttome/skin-cancer9-classesisic` on Kaggle), extended to satisfy the lab brief:

- **Table 1** — 8 transfer-learning backbones (AlexNet, VGG16, VGG19, ResNet18/50/101, DenseNet121, EfficientNet-B0)
- **Table 2** — Classical ML classifiers (Logistic Regression, Decision Tree, Random Forest, KNN, Linear SVM, RBF-SVM, XGBoost) trained on deep features extracted from the best backbone
- **Table 3** — Computational efficiency (parameters, model size, FLOPs, inference time) vs. accuracy

**Scope choices for a manageable runtime:**
- Only **5 manually chosen classes** are used out of the 9 available (`actinic keratosis`,
  `basal cell carcinoma`, `melanoma`, `nevus`, `pigmented benign keratosis`) — fixed by hand, not
  picked by image count — which comfortably clears the lab's 4-class minimum while cutting both
  data volume and GPU time versus using all 9.
- The official `Train`/`Test` directories are used as provided: the official `Test` folder is never
  touched or resplit, and a **stratified 90/10 split of the official `Train` folder** produces the
  validation set (same class balance in train and val).
- Epochs are capped, not fixed: a short 3-epoch head warm-up, then up to 10 fine-tuning epochs with
  **early stopping** (patience = 3 on validation loss, best weights restored). Training stops as soon as
  a model stops improving, so you're never running needless extra epochs — and never cut short before
  the model has actually converged.
- The backbone used as the Table 2 feature extractor is chosen by **validation accuracy**, never by
  test accuracy — the test set stays untouched until final reporting, so there's no leakage into
  model-selection decisions.



## 1. Setup

In [1]:
# Run once per session
!pip install -q kagglehub thop xgboost scikit-learn torchvision


In [2]:
import os, time, copy, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from thop import profile as thop_profile

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## 2. Download the dataset

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", path)

# Inspect the folder structure so we know how to point ImageFolder at it.
for root, dirs, files in os.walk(path):
    depth = root.replace(path, "").count(os.sep)
    if depth < 3:
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)" if files else "")


Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic
skin-cancer9-classesisic/ 
  Skin cancer ISIC The International Skin Imaging Collaboration/ 
    Test/ 
    Train/ 


**After running the cell above, check the printed tree.**
This dataset is normally organised as:

```
<path>/Skin cancer ISIC The International Skin Imaging Collaboration/
    Train/
        actinic keratosis/
        basal cell carcinoma/
        ... (9 class folders)
    Test/
        actinic keratosis/
        ...
```

If your printed structure differs, edit `TRAIN_DIR` / `TEST_DIR` below to match it — the rest of the
notebook only assumes standard `ImageFolder`-compatible class subfolders (minimum 4 classes; this
dataset has 9, which satisfies the lab requirement).


In [4]:
BASE_DIR = os.path.join(path, "Skin cancer ISIC The International Skin Imaging Collaboration")
TRAIN_DIR = os.path.join(BASE_DIR, "Train")
TEST_DIR  = os.path.join(BASE_DIR, "Test")

assert os.path.isdir(TRAIN_DIR), f"Adjust TRAIN_DIR — not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR),  f"Adjust TEST_DIR — not found: {TEST_DIR}"

ALL_CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
print(f"{len(ALL_CLASS_NAMES)} classes available:", ALL_CLASS_NAMES)


9 classes available: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']


## 2b. Manually selected classes (fixed — not data-driven)

The lab uses only 5 of the dataset's 9 classes to keep training time manageable. The 5 classes are fixed by hand (not chosen by image count) and their order below also fixes the label indices used throughout the rest of the notebook.

In [5]:
SELECTED_CLASSES = [
    "actinic keratosis",
    "basal cell carcinoma",
    "melanoma",
    "nevus",
    "pigmented benign keratosis",
]

missing = [c for c in SELECTED_CLASSES if c not in ALL_CLASS_NAMES]
assert not missing, f"These class folders were not found in TRAIN_DIR: {missing}"

CLASS_TO_IDX = {c: i for i, c in enumerate(SELECTED_CLASSES)}  # order preserved, not sorted
NUM_CLASSES = len(SELECTED_CLASSES)
print(f"Using {NUM_CLASSES} manually selected classes (fixed label order):")
for c, i in CLASS_TO_IDX.items():
    print(f"  {i}: {c}")


def filter_imagefolder_to_classes(dataset, class_to_idx):
    # Keep only samples whose class is in class_to_idx, remapped to the shared label space.
    old_idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}
    new_samples = [
        (path, class_to_idx[old_idx_to_class[old_label]])
        for path, old_label in dataset.samples
        if old_idx_to_class[old_label] in class_to_idx
    ]
    dataset.samples = new_samples
    dataset.imgs = new_samples
    dataset.targets = [s[1] for s in new_samples]
    dataset.classes = list(class_to_idx.keys())
    dataset.class_to_idx = class_to_idx
    return dataset


Using 5 manually selected classes (fixed label order):
  0: actinic keratosis
  1: basal cell carcinoma
  2: melanoma
  3: nevus
  4: pigmented benign keratosis


## 3. Data loading, preprocessing & augmentation

Mirrors the paper's pre-processing (resize/crop, rescale to [0,1], normalisation) and
augmentation (flips, shifts, zoom, shear) — implemented with `torchvision.transforms`.
Standard ImageNet input size (224×224) is used since all 8 backbones are ImageNet-pretrained.

In [6]:
IMG_SIZE = 224
BATCH_SIZE = 32

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), shear=10, scale=(0.8, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
test_ds       = datasets.ImageFolder(TEST_DIR, transform=eval_tfms)

# Keep only the 5 selected classes, remapped to labels 0..4 consistently across splits
full_train_ds = filter_imagefolder_to_classes(full_train_ds, CLASS_TO_IDX)
test_ds       = filter_imagefolder_to_classes(test_ds, CLASS_TO_IDX)

# Stratified 90/10 split of the official Train directory (the official Test directory is
# never touched or resplit — it's reserved for final evaluation only).
# Two ImageFolder instances (identical file order, different transforms) let train/val each
# get the right transform pipeline while sharing the same underlying image list and indices.
train_full_ds = full_train_ds  # augmented transforms (train_tfms)
val_full_ds = filter_imagefolder_to_classes(
    datasets.ImageFolder(TRAIN_DIR, transform=eval_tfms), CLASS_TO_IDX
)  # eval transforms (no augmentation)

targets = train_full_ds.targets
indices = np.arange(len(targets))
train_idx, val_idx = train_test_split(
    indices, test_size=0.10, stratify=targets, random_state=SEED
)

train_ds = Subset(train_full_ds, train_idx)  # augmented
val_ds   = Subset(val_full_ds, val_idx)      # eval-only, same underlying images as train_idx's complement

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


Train: 1572 | Val: 175 | Test: 80


## 4. Model factory

Builds each backbone with ImageNet weights and replaces the final classification layer for `NUM_CLASSES`. Returns the model plus the name of its final-layer attribute (needed later for feature extraction in Table 2).

In [7]:
def build_model(name: str, num_classes: int):
    name = name.lower()
    if name == "alexnet":
        m = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        head_attr = ("classifier", 6)
    elif name == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        head_attr = ("classifier", 6)
    elif name == "vgg19":
        m = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        head_attr = ("classifier", 6)
    elif name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        head_attr = ("fc", None)
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        head_attr = ("fc", None)
    elif name == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        head_attr = ("fc", None)
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        head_attr = ("classifier", None)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        head_attr = ("classifier", 1)
    else:
        raise ValueError(f"Unknown model: {name}")
    return m, head_attr


def set_backbone_trainable(model, trainable: bool, head_attr):
    # Freeze/unfreeze every param except the final classification layer.
    for p in model.parameters():
        p.requires_grad = trainable
    attr, idx = head_attr
    head = getattr(model, attr)
    if idx is not None:
        head = head[idx]
    for p in head.parameters():
        p.requires_grad = True


MODEL_NAMES = ["alexnet", "vgg16", "vgg19", "resnet18", "resnet50",
               "resnet101", "densenet121", "efficientnet_b0"]


## 5. Train / evaluate loop (two-phase fine-tuning, as in the paper)

`EarlyStopper` tracks validation loss during fine-tuning and stops the moment it stops improving (restoring the best-seen weights) — this is what keeps epoch count 'just enough': a hard cap (`ft_epochs`) prevents runaway training, and early stopping prevents wasted epochs once the model has converged.

In [8]:
class EarlyStopper:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


In [9]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


def train_model(name, num_classes, feat_epochs=3, ft_epochs=10, lr_head=1e-3, lr_ft=1e-4,
                 ft_patience=3):
    print(f"\n===== Training {name} =====")
    model, head_attr = build_model(name, num_classes)
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    # Phase 1: freeze backbone, train the new head only (short — it's just a warm-up)
    set_backbone_trainable(model, False, head_attr)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr_head)
    for ep in range(feat_epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, opt)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        print(f"[head]  epoch {ep+1}/{feat_epochs}  train_acc={tr_acc:.3f}  val_acc={val_acc:.3f}")

    # Phase 2: unfreeze everything, fine-tune at a lower LR — capped at ft_epochs, but
    # early stopping (patience=ft_patience on val loss) exits as soon as it stops helping.
    set_backbone_trainable(model, True, head_attr)
    opt = optim.Adam(model.parameters(), lr=lr_ft)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, factor=(0.1) ** 0.5, patience=2)
    stopper = EarlyStopper(patience=ft_patience)
    for ep in range(ft_epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, opt)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)
        print(f"[ft]    epoch {ep+1}/{ft_epochs}  train_acc={tr_acc:.3f}  val_acc={val_acc:.3f}")
        if stopper.step(val_loss, model):
            print(f"  early stopping (no val-loss improvement for {ft_patience} epochs)")
            break
    stopper.restore(model)

    # Final validation accuracy at the restored (best) weights — used later to pick the
    # feature-extractor backbone for Table 2 without ever looking at the test set.
    _, best_val_acc = run_epoch(model, val_loader, criterion)

    return model, head_attr, best_val_acc


@torch.no_grad()
def evaluate_model(model, loader, num_classes):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    for x, y in loader:
        x = x.to(DEVICE)
        out = model(x)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(probs.argmax(1))
        all_labels.append(y.numpy())
    probs = np.concatenate(all_probs)
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="macro", zero_division=0)
    rec = recall_score(labels, preds, average="macro", zero_division=0)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")  # e.g. a class missing from the test split
    return dict(Accuracy=acc * 100, Precision=prec * 100, Recall=rec * 100,
                F1_Score=f1 * 100, AUC=auc * 100)


## 6. Table 1 — Transfer Learning Models Comparison

**Tip:** if this is too slow on your GPU/time budget, cut `feat_epochs`/`ft_epochs`, or temporarily shrink `MODEL_NAMES` while you debug, then run the full list for your submitted results.

In [10]:
table1_rows = {}
trained_models = {}      # keep trained weights + head_attr for later reuse (Table 2/3)
val_accuracies = {}      # validation accuracy per backbone — used to pick the Table 2 feature
                          # extractor WITHOUT looking at the test set (no leakage)

for name in MODEL_NAMES:
    model, head_attr, val_acc = train_model(name, NUM_CLASSES)
    metrics = evaluate_model(model, test_loader, NUM_CLASSES)  # test set: final reporting only
    table1_rows[name] = metrics
    trained_models[name] = (model, head_attr)
    val_accuracies[name] = val_acc
    print(name, metrics, f"(val_acc={val_acc:.3f})")

table1_df = pd.DataFrame(table1_rows).T
table1_df.index.name = "Model"
table1_df = table1_df.round(2)
table1_df



===== Training alexnet =====
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 168MB/s]


[head]  epoch 1/3  train_acc=0.476  val_acc=0.617
[head]  epoch 2/3  train_acc=0.561  val_acc=0.594
[head]  epoch 3/3  train_acc=0.605  val_acc=0.600
[ft]    epoch 1/10  train_acc=0.625  val_acc=0.709
[ft]    epoch 2/10  train_acc=0.713  val_acc=0.749
[ft]    epoch 3/10  train_acc=0.720  val_acc=0.703
[ft]    epoch 4/10  train_acc=0.758  val_acc=0.697
[ft]    epoch 5/10  train_acc=0.782  val_acc=0.783
[ft]    epoch 6/10  train_acc=0.785  val_acc=0.777
[ft]    epoch 7/10  train_acc=0.796  val_acc=0.714
[ft]    epoch 8/10  train_acc=0.784  val_acc=0.749
  early stopping (no val-loss improvement for 3 epochs)
alexnet {'Accuracy': 57.49999999999999, 'Precision': 60.973905351614334, 'Recall': 57.49999999999999, 'F1_Score': 54.28571428571429, 'AUC': np.float64(85.859375)} (val_acc=0.783)

===== Training vgg16 =====
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 149MB/s]


[head]  epoch 1/3  train_acc=0.398  val_acc=0.577
[head]  epoch 2/3  train_acc=0.488  val_acc=0.560
[head]  epoch 3/3  train_acc=0.499  val_acc=0.583
[ft]    epoch 1/10  train_acc=0.478  val_acc=0.594
[ft]    epoch 2/10  train_acc=0.634  val_acc=0.709
[ft]    epoch 3/10  train_acc=0.702  val_acc=0.737
[ft]    epoch 4/10  train_acc=0.765  val_acc=0.743
[ft]    epoch 5/10  train_acc=0.753  val_acc=0.789
[ft]    epoch 6/10  train_acc=0.792  val_acc=0.771
[ft]    epoch 7/10  train_acc=0.795  val_acc=0.709
[ft]    epoch 8/10  train_acc=0.798  val_acc=0.789
[ft]    epoch 9/10  train_acc=0.787  val_acc=0.817
[ft]    epoch 10/10  train_acc=0.798  val_acc=0.760
vgg16 {'Accuracy': 63.74999999999999, 'Precision': 66.82019704433498, 'Recall': 63.74999999999999, 'F1_Score': 59.70766920766921, 'AUC': np.float64(89.8046875)} (val_acc=0.817)

===== Training vgg19 =====
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

100%|██████████| 30.8M/30.8M [00:00<00:00, 172MB/s]


[head]  epoch 1/3  train_acc=0.433  val_acc=0.571
[head]  epoch 2/3  train_acc=0.597  val_acc=0.611
[head]  epoch 3/3  train_acc=0.633  val_acc=0.640
[ft]    epoch 1/10  train_acc=0.706  val_acc=0.771
[ft]    epoch 2/10  train_acc=0.799  val_acc=0.754
[ft]    epoch 3/10  train_acc=0.819  val_acc=0.777
[ft]    epoch 4/10  train_acc=0.849  val_acc=0.789
[ft]    epoch 5/10  train_acc=0.865  val_acc=0.771
[ft]    epoch 6/10  train_acc=0.867  val_acc=0.771
[ft]    epoch 7/10  train_acc=0.895  val_acc=0.806
[ft]    epoch 8/10  train_acc=0.892  val_acc=0.760
  early stopping (no val-loss improvement for 3 epochs)
densenet121 {'Accuracy': 58.75, 'Precision': 67.38528138528139, 'Recall': 58.75, 'F1_Score': 53.954792103598656, 'AUC': np.float64(87.8515625)} (val_acc=0.771)

===== Training efficientnet_b0 =====
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 110MB/s] 


[head]  epoch 1/3  train_acc=0.471  val_acc=0.566
[head]  epoch 2/3  train_acc=0.621  val_acc=0.606
[head]  epoch 3/3  train_acc=0.652  val_acc=0.634
[ft]    epoch 1/10  train_acc=0.688  val_acc=0.709
[ft]    epoch 2/10  train_acc=0.753  val_acc=0.737
[ft]    epoch 3/10  train_acc=0.780  val_acc=0.760
[ft]    epoch 4/10  train_acc=0.805  val_acc=0.777
[ft]    epoch 5/10  train_acc=0.826  val_acc=0.766
[ft]    epoch 6/10  train_acc=0.858  val_acc=0.766
[ft]    epoch 7/10  train_acc=0.841  val_acc=0.783
[ft]    epoch 8/10  train_acc=0.879  val_acc=0.800
[ft]    epoch 9/10  train_acc=0.884  val_acc=0.783
[ft]    epoch 10/10  train_acc=0.883  val_acc=0.800
  early stopping (no val-loss improvement for 3 epochs)
efficientnet_b0 {'Accuracy': 60.0, 'Precision': 73.54901960784315, 'Recall': 60.0, 'F1_Score': 57.01073896863371, 'AUC': np.float64(90.60546875)} (val_acc=0.783)


,Accuracy,Precision,Recall,F1_Score,AUC
Model,,,,,
alexnet,57.50,60.97,57.50,54.29,85.86
vgg16,63.75,66.82,63.75,59.71,89.80
vgg19,65.00,68.10,65.00,60.82,92.85
resnet18,62.50,69.79,62.50,59.05,87.68
resnet50,70.00,75.97,70.00,67.64,88.89
resnet101,63.75,74.11,63.75,62.11,90.84
densenet121,58.75,67.39,58.75,53.95,87.85
efficientnet_b0,60.00,73.55,60.00,57.01,90.61


## 7. Table 2 — Comparison of Different Classifiers on Deep Features

We pick the **best backbone from Table 1** as the frozen feature extractor, using each
backbone's **validation accuracy** (not test accuracy) for the selection — the test set is reserved
purely for final reporting and must never influence which model gets picked. We then strip the
selected backbone's final classification layer and use the penultimate (pooled) activations as a
fixed-length feature vector for every image. Classical ML classifiers are then trained on those features.

In [11]:
best_model_name = max(val_accuracies, key=val_accuracies.get)
print("Best backbone for feature extraction (chosen by validation accuracy, not test):",
      best_model_name, f"(val_acc={val_accuracies[best_model_name]:.3f})")

best_model, head_attr = trained_models[best_model_name]
best_model.to(DEVICE).eval()

feature_extractor = copy.deepcopy(best_model)
attr, idx = head_attr
if idx is not None:
    getattr(feature_extractor, attr)[idx] = nn.Identity()
else:
    setattr(feature_extractor, attr, nn.Identity())
feature_extractor.to(DEVICE).eval()

@torch.no_grad()
def extract_features(loader):
    feats, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        f = feature_extractor(x)
        f = torch.flatten(f, 1)
        feats.append(f.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

# Training-subset images only (train_idx), with eval transforms (no augmentation) — the
# validation subset (val_idx) is excluded so it never leaks into classifier fitting.
clean_train_ds = Subset(val_full_ds, train_idx)

clean_train_loader = DataLoader(
    clean_train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

X_train, y_train = extract_features(clean_train_loader)
X_test, y_test = extract_features(test_loader)
print("Feature shapes:", X_train.shape, X_test.shape)


Best backbone for feature extraction (chosen by validation accuracy, not test): vgg16 (val_acc=0.817)
Feature shapes: (1572, 4096) (80, 4096)


In [12]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Linear SVM": SVC(kernel="linear", probability=True, random_state=SEED),
    "RBF-SVM": SVC(kernel="rbf", probability=True, random_state=SEED),
    "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=SEED),
}

table2_rows = {}
for clf_name, clf in classifiers.items():
    print("Training:", clf_name)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average="macro", zero_division=0)
    rec = recall_score(y_test, preds, average="macro", zero_division=0)
    f1 = f1_score(y_test, preds, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(y_test, probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")

    table2_rows[clf_name] = dict(Feature_Extractor=f"Deep Features ({best_model_name})",
                                  Accuracy=acc * 100, Precision=prec * 100,
                                  Recall=rec * 100, F1_Score=f1 * 100, AUC=auc * 100)

table2_df = pd.DataFrame(table2_rows).T
table2_df.index.name = "Classifier"
num_cols = ["Accuracy", "Precision", "Recall", "F1_Score", "AUC"]
table2_df[num_cols] = table2_df[num_cols].astype(float).round(2)
table2_df


Training: Logistic Regression
Training: Decision Tree
Training: Random Forest
Training: K-Nearest Neighbors (KNN)
Training: Linear SVM
Training: RBF-SVM
Training: XGBoost


,Feature_Extractor,Accuracy,Precision,Recall,F1_Score,AUC
Classifier,,,,,,
Logistic Regression,Deep Features (vgg16),55.00,57.84,55.00,48.07,88.28
Decision Tree,Deep Features (vgg16),60.00,58.12,60.00,55.05,74.99
Random Forest,Deep Features (vgg16),56.25,59.77,56.25,49.48,93.29
K-Nearest Neighbors (KNN),Deep Features (vgg16),58.75,63.82,58.75,53.60,84.78
Linear SVM,Deep Features (vgg16),57.50,60.95,57.50,52.49,88.44
RBF-SVM,Deep Features (vgg16),65.00,73.18,65.00,60.21,92.54
XGBoost,Deep Features (vgg16),57.50,59.20,57.50,51.16,90.29


## 8. Table 3 — Computational Efficiency Comparison

In [13]:
def get_model_size_mb(model):
    tmp_path = "/tmp/_tmp_model.pt"
    torch.save(model.state_dict(), tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 ** 2)
    os.remove(tmp_path)
    return size_mb


def measure_inference_time_ms(model, n_runs=50, warmup=10):
    model.eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    with torch.no_grad():
        for _ in range(warmup):
            model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(n_runs):
            model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        end = time.time()
    return (end - start) / n_runs * 1000  # ms per image


table3_rows = {}
for name in MODEL_NAMES:
    model, _ = trained_models[name]
    model.to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters()) / 1e6  # millions
    size_mb = get_model_size_mb(model)

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    macs, _ = thop_profile(copy.deepcopy(model), inputs=(dummy,), verbose=False)
    flops_g = (2 * macs) / 1e9  # 1 MAC ~= 2 FLOPs

    inf_time = measure_inference_time_ms(model)

    table3_rows[name] = dict(
        Parameters_M=n_params,
        Model_Size_MB=size_mb,
        FLOPs_G=flops_g,
        Inference_Time_ms=inf_time,
        Accuracy=table1_df.loc[name, "Accuracy"],
    )

table3_df = pd.DataFrame(table3_rows).T
table3_df.index.name = "Model"
table3_df = table3_df.round(2)
table3_df


,Parameters_M,Model_Size_MB,FLOPs_G,Inference_Time_ms,Accuracy
Model,,,,,
alexnet,57.02,217.54,1.42,2.08,57.50
vgg16,134.28,512.25,30.93,9.82,63.75
vgg19,139.59,532.51,39.26,11.49,65.00
resnet18,11.18,42.72,3.65,3.69,62.50
resnet50,23.52,90.02,8.26,8.65,70.00
resnet101,42.51,162.76,15.73,17.85,63.75
densenet121,6.96,27.13,5.79,15.58,58.75
efficientnet_b0,4.01,15.60,0.83,8.31,60.00


## 9. Export results (ready to paste into `Task_01.docx`)

In [14]:
os.makedirs("results", exist_ok=True)
table1_df.to_csv("results/table1_transfer_learning_models.csv")
table2_df.to_csv("results/table2_classifiers.csv")
table3_df.to_csv("results/table3_computational_efficiency.csv")

print("Table 1\n", table1_df.to_markdown())
print("\nTable 2\n", table2_df.to_markdown())
print("\nTable 3\n", table3_df.to_markdown())


Table 1
 | Model           |   Accuracy |   Precision |   Recall |   F1_Score |   AUC |
|:----------------|-----------:|------------:|---------:|-----------:|------:|
| alexnet         |      57.5  |       60.97 |    57.5  |      54.29 | 85.86 |
| vgg16           |      63.75 |       66.82 |    63.75 |      59.71 | 89.8  |
| vgg19           |      65    |       68.1  |    65    |      60.82 | 92.85 |
| resnet18        |      62.5  |       69.79 |    62.5  |      59.05 | 87.68 |
| resnet50        |      70    |       75.97 |    70    |      67.64 | 88.89 |
| resnet101       |      63.75 |       74.11 |    63.75 |      62.11 | 90.84 |
| densenet121     |      58.75 |       67.39 |    58.75 |      53.95 | 87.85 |
| efficientnet_b0 |      60    |       73.55 |    60    |      57.01 | 90.61 |

Table 2
 | Classifier                | Feature_Extractor     |   Accuracy |   Precision |   Recall |   F1_Score |   AUC |
|:--------------------------|:----------------------|-----------:|------------